In [2]:
import os
import time
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from torch.utils.tensorboard import SummaryWriter

log_dir = os.path.join("runs",f"regression_mlp_{int(time.time())}")
writer = SummaryWriter(log_dir=log_dir)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f" TensorBoard telemetry active. Logging to: {log_dir}")
print(f" Core compute engine target: {device}")

 TensorBoard telemetry active. Logging to: runs\regression_mlp_1782488010
 Core compute engine target: cuda


In [4]:
torch.manual_seed(42)
X_raw = torch.rand(5000,13)
y_raw = torch.sum(X_raw[:, :5] * 15.0, dim=1) + torch.sin(X_raw[:, 5]) * 10.0 + torch.randn(5000) * 2.0
y_raw = y_raw.unsqueeze(1)

split = int(.8*len(X_raw))
X_train, X_test = X_raw[:split], X_raw[split:]
y_train, y_test = y_raw[:split], y_raw[split:]

train_dataset = TensorDataset(X_train,y_train)
test_dataset = TensorDataset(X_test,y_test)

BATCH_SIZE = 64
train_loader = DataLoader(train_dataset,batch_size = BATCH_SIZE,shuffle=True,drop_last=True)
test_loader = DataLoader(test_dataset,batch_size=BATCH_SIZE,shuffle=False)
print(f"Data Batches Initialized. Training Stream Size: {len(train_loader)} steps per epoch.")

Data Batches Initialized. Training Stream Size: 62 steps per epoch.


In [5]:
model = nn.Sequential(
    nn.Linear(in_features=13,out_features=128),
    nn.ReLU(),
    nn.Linear(in_features=128,out_features=64),
    nn.ReLU(),
    nn.Linear(in_features=64,out_features=1)
)

model=model.to(device)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(),lr=1e-3)

dummy_input = torch.rand(1,13).to(device)
writer.add_graph(model,dummy_input)
print(model)

Sequential(
  (0): Linear(in_features=13, out_features=128, bias=True)
  (1): ReLU()
  (2): Linear(in_features=128, out_features=64, bias=True)
  (3): ReLU()
  (4): Linear(in_features=64, out_features=1, bias=True)
)


In [7]:
EPOCHS = 15
global_step = 0

print("Launching Production Regression MLP Loop...\n")

for epoch in range(EPOCHS):
    model.train()
    running_train_loss = 0.0
    
    for batch_X, batch_y in train_loader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)
        
        optimizer.zero_grad(set_to_none=True)
        predictions = model(batch_X)
        loss = criterion(predictions, batch_y)
        loss.backward()
        optimizer.step()
        
        running_train_loss += loss.item()
        
        # Log training loss values per step for fine-grained analysis
        writer.add_scalar("Loss/Train_Step", loss.item(), global_step)
        global_step += 1
        
    # Evaluate model performance against the holdout validation set
    model.eval()
    running_test_loss = 0.0
    with torch.no_grad():
        for test_X, test_y in test_loader:
            test_X, test_y = test_X.to(device), test_y.to(device)
            test_preds = model(test_X)
            running_test_loss += criterion(test_preds, test_y).item()
            
    epoch_train_loss = running_train_loss / len(train_loader)
    epoch_test_loss  = running_test_loss / len(test_loader)
    
    # Send metrics directly to TensorBoard charts
    writer.add_scalar("Loss/Epoch_Train", epoch_train_loss, epoch)
    writer.add_scalar("Loss/Epoch_Test", epoch_test_loss, epoch)
    
    # Log layer weight distributions to track gradient health
    for name, param in model.named_parameters():
        if 'weight' in name:
            writer.add_histogram(f"Weights/{name}", param, epoch)
            
    print(f" Epoch [{epoch+1}/{EPOCHS}]")
    print(f"   ↳ Mean Squared Error (Train MSE): {epoch_train_loss:.4f}")
    print(f"   ↳ Mean Squared Error (Test MSE) : {epoch_test_loss:.4f}\n")

# Flush and close the TensorBoard data stream
writer.close()
print("Training complete. Telemetry buffers successfully closed.")

Launching Production Regression MLP Loop...

 Epoch [1/15]
   ↳ Mean Squared Error (Train MSE): 4.2539
   ↳ Mean Squared Error (Test MSE) : 4.4138

 Epoch [2/15]
   ↳ Mean Squared Error (Train MSE): 4.2185
   ↳ Mean Squared Error (Test MSE) : 4.3912

 Epoch [3/15]
   ↳ Mean Squared Error (Train MSE): 4.1956
   ↳ Mean Squared Error (Test MSE) : 4.3832

 Epoch [4/15]
   ↳ Mean Squared Error (Train MSE): 4.2146
   ↳ Mean Squared Error (Test MSE) : 4.3476

 Epoch [5/15]
   ↳ Mean Squared Error (Train MSE): 4.1586
   ↳ Mean Squared Error (Test MSE) : 4.4216

 Epoch [6/15]
   ↳ Mean Squared Error (Train MSE): 4.1554
   ↳ Mean Squared Error (Test MSE) : 4.3792

 Epoch [7/15]
   ↳ Mean Squared Error (Train MSE): 4.1330
   ↳ Mean Squared Error (Test MSE) : 4.3678

 Epoch [8/15]
   ↳ Mean Squared Error (Train MSE): 4.1963
   ↳ Mean Squared Error (Test MSE) : 4.3302

 Epoch [9/15]
   ↳ Mean Squared Error (Train MSE): 4.1425
   ↳ Mean Squared Error (Test MSE) : 4.3090

 Epoch [10/15]
   ↳ Mean Squ